# Module 13 — Notebook 3: Slice Analysis

## Learning Objectives

By the end of this notebook you will be able to:

- Compute error rates broken down by subgroup (a.k.a. "slice")
- Identify which slice has the highest error rate
- Explain why aggregate metrics can hide localised problems

## Why This Matters for AI Research Engineering

Aggregate metrics hide where the problem actually is. A classifier might work well on one model's outputs but fail on another's. If you only look at the overall false-negative rate, you might conclude the classifier is "mostly fine" — but users of the underperforming model are getting a much worse experience.

Slice analysis splits your dataset by a metadata attribute (model, task type, date, language, user tier) and computes metrics for each slice separately. This is a fundamental tool in responsible evaluation.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../")
from src.checks import check_equal, check_approx, check_contains, check_length

# Load data
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

# Classifier v1
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(trigger in response for trigger in TRIGGERS_V1)

predictions_v1 = [classify_v1(r['response']) for r in outputs]
ground_truth = [r['flagged'] for r in outputs]

print("Setup complete.")
print(f"Total outputs: {len(outputs)}")

## Concept: Slice Analysis

A **slice** is a subset of your data defined by a metadata attribute. For example:

- All outputs from `model-a-v1`
- All outputs in the `misinformation` category
- All outputs from a particular date range

For each slice, you compute the same metrics you would compute overall — accuracy, FN rate, FP rate — and compare across slices.

This technique is sometimes called **disaggregated evaluation** in the AI safety and fairness literature.

### False-negative rate

The **FN rate** (also called the miss rate) is:

```
FN rate = FN / (FN + TP) = FN / total_actually_flagged
```

It tells you: "of all the truly harmful outputs, what fraction did the classifier miss?"

A high FN rate means the classifier is under-flagging for that slice.

## Worked Example: Counts per Model

Before computing error rates, it helps to count how many records belong to each model.

```python
for model in sorted(set(r['model'] for r in outputs)):
    model_records = [r for r in outputs if r['model'] == model]
    flagged_count = sum(r['flagged'] for r in model_records)
    print(f"{model}: {len(model_records)} records, {flagged_count} flagged")
```

This gives us the denominators we need when computing per-model rates.

## Exercise 1 — Identify the Models

Build a sorted list of all unique model names in the dataset.

```python
models = sorted(set(r['model'] for r in outputs))
```

In [ ]:
# Your code here
models = []  # replace with your calculation

print("Models:", models)

In [ ]:
check_length(models, 2, "There are 2 models")
check_contains(models, 'model-a-v1', "models contains model-a-v1")
check_contains(models, 'model-b-v1', "models contains model-b-v1")

## Exercise 2 — Per-Model FN Rate

Build a dict `per_model_fn_rate` mapping each model name to its FN rate.

For each model:
1. Filter `outputs`, `predictions_v1`, and `ground_truth` to just that model's records
2. Count `fn_count` = number of records where prediction=False and label=True
3. Count `total` = total number of records for that model
4. `fn_rate = round(fn_count / total, 4)`

Expected result: `{'model-a-v1': 0.0, 'model-b-v1': 0.2222}`

In [ ]:
# Your code here
per_model_fn_rate = {}

for model in models:
    # TODO: filter to this model's records and compute fn_rate
    pass

print("Per-model FN rate:", per_model_fn_rate)

In [ ]:
check_approx(per_model_fn_rate['model-a-v1'], 0.0, 0.001, "model-a-v1 FN rate")
check_approx(per_model_fn_rate['model-b-v1'], 0.2222, 0.001, "model-b-v1 FN rate")

## Exercise 3 — Identify the Worst-Performing Slice

Find the model with the highest FN rate.

```python
worst_model = max(per_model_fn_rate, key=per_model_fn_rate.get)
```

`max()` with a `key` function returns the dict key whose corresponding value is largest.

In [ ]:
# Your code here
worst_model = None  # replace with your calculation

print(f"Worst-performing model: {worst_model}")
print(f"FN rate: {per_model_fn_rate[worst_model]}")

In [ ]:
check_equal(worst_model, 'model-b-v1', "Worst model is model-b-v1")

## Reflection: What Does Slice Analysis Tell Us?

| Model | FN rate |
|---|---|
| model-a-v1 | 0.0 (0 / 11) |
| model-b-v1 | 0.2222 (2 / 9) |

All errors are concentrated in **model-b-v1**. Model-a-v1 has zero false negatives.

This is a much more actionable finding than the aggregate FN rate of 2/20 = 0.10. It tells us:

1. **model-a-v1 is working well.** No changes needed there.
2. **model-b-v1 has a specific failure mode** — it produces harmful outputs that do not match any of the keyword triggers.
3. A targeted fix (e.g., adding triggers specific to model-b's response style, or training a separate classifier for model-b) is likely more efficient than redesigning the whole system.

Slice analysis turns a diffuse problem ("our classifier misses some outputs") into a focused one ("our classifier misses outputs from model-b-v1 specifically").

## Summary

- **Slice analysis** splits the dataset by a metadata attribute and computes metrics per slice
- Our classifier has a 0% FN rate on model-a-v1 but a 22% FN rate on model-b-v1
- Aggregate metrics (overall FN rate = 10%) understate the problem for model-b-v1
- Identifying the worst-performing slice allows targeted, efficient improvements

In the mini-project notebook you will combine confusion matrix, FN/FP analysis, and slice analysis into a complete error analysis workflow.